# Mini Project: Sentiment Assistant with BERT Fine-Tuning

This notebook fine-tunes **bert-base-uncased** on the IMDB Reviews dataset.

In [ ]:
# Run once in a fresh environment
!pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

In [ ]:
import platform
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python:", platform.python_version())
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## Load IMDB Dataset

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True
)
print(ds_info)
for text,label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250],"...\n")


## Tokenization

In [ ]:
MAX_LENGTH=256
BATCH_SIZE=16

tokenizer=BertTokenizer.from_pretrained("bert-base-uncased",do_lower_case=True)

def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text=review_input.decode("utf-8")
    elif hasattr(review_input,"numpy"):
        review_text=review_input.numpy().decode("utf-8")
    else:
        review_text=str(review_input)
    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text,label):
    encoded=tf.py_function(
        func=lambda t:list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32,tf.int32,tf.int32]
    )
    return {
        "input_ids":encoded[0],
        "attention_mask":encoded[1],
        "token_type_ids":encoded[2]
    },label

def prepare_dataset(dataset):
    return (dataset
            .map(tf_encode,num_parallel_calls=tf.data.AUTOTUNE)
            .shuffle(2000)
            .batch(BATCH_SIZE)
            .prefetch(tf.data.AUTOTUNE))

train_ds=prepare_dataset(ds_train)
test_ds=prepare_dataset(ds_test)


## Load and Fine-Tune BERT

In [ ]:
model=TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False
)

optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5,epsilon=1e-8)
loss_fn=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]
)

model.summary()


## Training

In [ ]:
EPOCHS=2

history=model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)


## Evaluation

In [ ]:
eval_metrics=model.evaluate(test_ds)

print(f"Test Loss: {eval_metrics[0]:.4f}")
print(f"Test Accuracy: {eval_metrics[1]:.4f}")


## Inference

In [ ]:
def predict_sentiment(text:str):
    encoded=tokenizer(
        text,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )
    outputs=model(encoded)
    probs=tf.nn.softmax(outputs.logits,axis=-1).numpy()[0]
    label="Positive" if np.argmax(probs)==1 else "Negative"
    return label,float(np.max(probs))

custom_sentence="The onboarding emails were confusing, but the agent fixed everything politely."
label,confidence=predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")


# Reflection

### 1. What lever most improved the results?
Fine-tuning a pre-trained BERT model with a small learning rate (2e-5) provided the greatest improvement.

### 2. Where would you add guardrails before deployment?
- Confidence threshold
- Human review for uncertain predictions
- Bias monitoring
- Data drift monitoring

### 3. Which stakeholders benefit the most?
- Support Lead
- Product Manager
- Compliance Officer
